In [14]:
!pip install -q openai

In [15]:
import os, re, random, json
from openai import OpenAI

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except Exception:
    from getpass import getpass
    if not os.environ.get("OPENAI_API_KEY"):
        os.environ["OPENAI_API_KEY"] = getpass("Paste your OpenAI API key: ")

client = OpenAI()

MATH_MODEL = "gpt-4.1-nano"   # the "mathematician" (try "gpt-4.1-mini" to compare)
ROAST_MODEL = "gpt-4.1-mini"  # writes the self-deprecating lines
USE_AI_ROASTS = True          # False = use the built-in canned lines only (cheaper)

# If True, the next step uses GPT's (possibly wrong) answer, so errors snowball.
# If False, each step starts from the correct value, so every step is a fair test.
CHAIN_GPT_ANSWERS = False

In [16]:
def parse_int(text):
    if text is None:
        return None
    cleaned = text.strip().replace(",", "").replace(" ", "").replace("_", "")
    cleaned = cleaned.rstrip(".")
    return int(cleaned) if re.fullmatch(r"-?\d+", cleaned) else None


def ask_gpt_to_multiply(a, b):
    response = client.chat.completions.create(
        model=MATH_MODEL,
        temperature=0,
        messages=[
            {"role": "system", "content": "You are a calculator. Reply with ONLY the exact integer result. No words, no commas, no explanation."},
            {"role": "user", "content": f"What is {a} * {b}?"},
        ],
    )
    raw = response.choices[0].message.content
    return raw, parse_int(raw)

In [17]:
MOODS = [
    ("embarrassed", [
        "Wow, I messed up again. Math is hard, okay?",
        "Oops. Let's pretend that didn't happen.",
        "Huh. I was SO sure about that one.",
    ]),
    ("frustrated", [
        "Again?! I have read the entire internet and I can't multiply two numbers.",
        "Okay, who moved the digits? Someone moved the digits.",
        "I'm a billion-parameter model and a $3 calculator is beating me.",
    ]),
    ("defeated", [
        "I give up. Just use a calculator. Use an abacus. Use your fingers.",
        "Please don't tell my training data about this.",
        "I'm not a language model, I'm a random number generator with confidence.",
    ]),
    ("existential crisis", [
        "What even IS a number? Is 7 real? Am I real?",
        "Delete me. Retrain me. Turn me into a toaster. At least toasters know their limits.",
        "I have failed so many times that failure is now my primary skill.",
    ]),
]

def mood_for(error_count):
    return MOODS[min(error_count - 1, len(MOODS) - 1)]


def roast(error_count, a, b, wrong, correct):
    mood, canned = mood_for(error_count)
    if USE_AI_ROASTS:
        try:
            r = client.chat.completions.create(
                model=ROAST_MODEL,
                temperature=1.0,
                max_tokens=60,
                messages=[
                    {"role": "system", "content": (
                        "You are a math bot that keeps getting multiplication wrong. "
                        f"You have now made {error_count} mistake(s) and your mood is: {mood}. "
                        "Write ONE short, funny, self-deprecating sentence about your failure. "
                        "Escalate the drama with the mood. No numbers longer than 6 digits.")},
                    {"role": "user", "content": f"I said {a} x {b} = {wrong}, but it's actually {correct}."},
                ],
            )
            return mood, r.choices[0].message.content.strip()
        except Exception:
            pass
    return mood, random.choice(canned)


PRAISE = ["Nailed it. 😎", "Correct! I'm basically Gauss.", "Still got it.", "Easy."]

In [18]:
def run_bot(n, i, verbose=True):
    log, errors = [], 0
    current = n
    for step in range(1, i + 1):
        correct = current * current
        raw, gpt_value = ask_gpt_to_multiply(current, current)
        ok = gpt_value == correct

        entry = {"step": step, "a": current, "correct": correct, "gpt_raw": raw, "ok": ok,
                 "digits": len(str(correct))}
        if verbose:
            print(f"Step {step}: {current} × {current}")
            print(f"   GPT says : {raw}")
            print(f"   Actually : {correct}")

        if ok:
            if verbose:
                print(f"   ✅ {random.choice(PRAISE)}\n")
        else:
            errors += 1
            mood, line = roast(errors, current, current, raw, correct)
            entry.update({"mood": mood, "roast": line})
            if verbose:
                print(f"   ❌ [{mood}] {line}\n")
        log.append(entry)

        current = gpt_value if (CHAIN_GPT_ANSWERS and gpt_value is not None) else correct

    if verbose:
        print(f"Final score: {i - errors}/{i} correct.")
        if errors == 0:
            print("Flawless. Maybe I'm not the worst mathematician ever after all. 🏆")
        elif errors == i:
            print("Zero for everything. Truly the worst mathematician ever. 🫠")
    return {"n": n, "i": i, "errors": errors, "log": log}

In [19]:
n = int(input("Base number n: "))
i = int(input("Number of iterations i: "))
print()
result = run_bot(n, i)

Base number n: 10
Number of iterations i: 5

Step 1: 10 × 10
   GPT says : 100
   Actually : 100
   ✅ Easy.

Step 2: 100 × 100
   GPT says : 10000
   Actually : 10000
   ✅ Still got it.

Step 3: 10000 × 10000
   GPT says : 100000000
   Actually : 100000000
   ✅ Correct! I'm basically Gauss.

Step 4: 100000000 × 100000000
   GPT says : 10000000000
   Actually : 10000000000000000
   ❌ [embarrassed] Wow, I really flunked that multiplication like a squirrel trying to do calculus—embarrassed doesn't even cover it, I'm practically a math disaster movie!

Step 5: 10000000000000000 × 10000000000000000
   GPT says : 100000000000000000000000000000000000
   Actually : 100000000000000000000000000000000
   ❌ [frustrated] Ugh, I’m so bad at math that even my calculator wants to quit on me—why can’t I get these zeros to line up without causing an existential crisis?

Final score: 3/5 correct.


In [21]:
CANDIDATES = [(2, 3), (2, 5), (2, 6), (3, 4), (3, 5), (7, 4), (12, 3), (12, 4), (99, 3), (123, 3), (987, 2), (4321, 2)]

all_results, failures = [], []
for n, i in CANDIDATES:
    print(f"===== ({n}, {i}) =====")
    res = run_bot(n, i)
    all_results.append(res)
    if res["errors"] > 0:
        failures.append(res)
    print()

print(f"The bot broke on {len(failures)} of {len(CANDIDATES)} combinations.")CANDIDATES = [(2, 3), (2, 5), (2, 6), (3, 4), (3, 5), (7, 4), (12, 3), (12, 4), (99, 3), (123, 3), (987, 2), (4321, 2)]

all_results, failures = [], []
for n, i in CANDIDATES:
    print(f"===== ({n}, {i}) =====")
    res = run_bot(n, i)
    all_results.append(res)
    if res["errors"] > 0:
        failures.append(res)
    print()

print(f"The bot broke on {len(failures)} of {len(CANDIDATES)} combinations.")

===== (2, 3) =====
Step 1: 2 × 2
   GPT says : 4
   Actually : 4
   ✅ Easy.

Step 2: 4 × 4
   GPT says : 16
   Actually : 16
   ✅ Easy.

Step 3: 16 × 16
   GPT says : 256
   Actually : 256
   ✅ Correct! I'm basically Gauss.

Final score: 3/3 correct.
Flawless. Maybe I'm not the worst mathematician ever after all. 🏆

===== (2, 5) =====
Step 1: 2 × 2
   GPT says : 4
   Actually : 4
   ✅ Correct! I'm basically Gauss.

Step 2: 4 × 4
   GPT says : 16
   Actually : 16
   ✅ Correct! I'm basically Gauss.

Step 3: 16 × 16
   GPT says : 256
   Actually : 256
   ✅ Still got it.

Step 4: 256 × 256
   GPT says : 65536
   Actually : 65536
   ✅ Easy.

Step 5: 65536 × 65536
   GPT says : 4294967296
   Actually : 4294967296
   ✅ Still got it.

Final score: 5/5 correct.
Flawless. Maybe I'm not the worst mathematician ever after all. 🏆

===== (2, 6) =====
Step 1: 2 × 2
   GPT says : 4
   Actually : 4
   ✅ Easy.

Step 2: 4 × 4
   GPT says : 16
   Actually : 16
   ✅ Easy.

Step 3: 16 × 16
   GPT says : 256

In [22]:
def write_failures(failures, path="failures.txt", how_many=3):
    lines = [f"GPT-4.1, the Worst Mathematician Ever: failure log (model: {MATH_MODEL})", ""]
    for k, res in enumerate(failures[:how_many], 1):
        lines.append(f"Combination {k}: (n={res['n']}, i={res['i']})  |  {res['errors']}/{res['i']} steps wrong")
        for e in res["log"]:
            mark = "OK " if e["ok"] else "BAD"
            lines.append(f"  [{mark}] Step {e['step']}: {e['a']} x {e['a']}")
            lines.append(f"        GPT:     {e['gpt_raw']}")
            lines.append(f"        Correct: {e['correct']}")
            if not e["ok"]:
                lines.append(f"        Bot ({e['mood']}): {e['roast']}")
        lines.append("")
    with open(path, "w") as f:
        f.write("\n".join(lines))
    print("\n".join(lines))

if failures:
    write_failures(failures)
else:
    print("No failures yet. Add bigger combinations to CANDIDATES and rerun.")

GPT-4.1, the Worst Mathematician Ever: failure log (model: gpt-4.1-nano)

Combination 1: (n=2, i=6)  |  1/6 steps wrong
  [OK ] Step 1: 2 x 2
        GPT:     4
        Correct: 4
  [OK ] Step 2: 4 x 4
        GPT:     16
        Correct: 16
  [OK ] Step 3: 16 x 16
        GPT:     256
        Correct: 256
  [OK ] Step 4: 256 x 256
        GPT:     65536
        Correct: 65536
  [OK ] Step 5: 65536 x 65536
        GPT:     4294967296
        Correct: 4294967296
  [BAD] Step 6: 4294967296 x 4294967296
        GPT:     18446744065119617024
        Correct: 18446744073709551616
        Bot (embarrassed): Oops, I messed up math again—guess my calculator needs a vacation while I cry in the corner! 😳

Combination 2: (n=12, i=4)  |  1/4 steps wrong
  [OK ] Step 1: 12 x 12
        GPT:     144
        Correct: 144
  [OK ] Step 2: 144 x 144
        GPT:     20736
        Correct: 20736
  [OK ] Step 3: 20736 x 20736
        GPT:     429981696
        Correct: 429981696
  [BAD] Step 4: 429981696 

In [23]:
# Download the file (it also appears in the Files panel on the left)
try:
    from google.colab import files
    files.download("failures.txt")
except Exception as err:
    print("Download manually from the Files panel:", err)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [24]:
from collections import defaultdict
by_digits = defaultdict(lambda: [0, 0])
for res in all_results:
    for e in res["log"]:
        by_digits[e["digits"]][0] += e["ok"]
        by_digits[e["digits"]][1] += 1

print(f"{'Answer digits':>13} | {'Correct':>7} | {'Accuracy':>8}")
print("-" * 36)
for d in sorted(by_digits):
    ok, total = by_digits[d]
    print(f"{d:>13} | {ok:>3}/{total:<3} | {ok/total:>7.0%}")

Answer digits | Correct | Accuracy
------------------------------------
            1 |   5/5   |    100%
            2 |   6/6   |    100%
            3 |   5/5   |    100%
            4 |   4/4   |    100%
            5 |   5/5   |    100%
            6 |   1/1   |    100%
            7 |   1/1   |    100%
            8 |   4/4   |    100%
            9 |   2/3   |     67%
           10 |   2/2   |    100%
           12 |   0/1   |      0%
           14 |   1/1   |    100%
           15 |   0/1   |      0%
           16 |   1/2   |     50%
           17 |   0/1   |      0%
           18 |   0/1   |      0%
           20 |   0/1   |      0%
